# Base Place Recognition Pipeline 

Test Place Recognition on the 3DSSG dataset using `opr.pipelines`

In [1]:
import itertools
import shutil
from pathlib import Path
import json

import faiss
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go

from torchvision import transforms as T
from opr.datasets.itlp import ITLPCampus
#from opr.models.place_recognition import MinkLoc3D
from mmpr.inference import PlaceRecognitionPipeline, FaissFlatIndex, SequencePlaceRecognitionPipeline

from gsloc.inference.pr_infer import PRInferencer
from gsloc.models import graph_encoder as network
# from opr.pipelines.place_recognition import PlaceRecognitionPipeline

from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/home/kartashov_ga/projects/GSLoc/.venv/lib/python3.10/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error _ssl.c:1000: The handshake operation timed out>
  data = fetch_version_info()
2026-04-22 16:43:36.138 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


## Create dataset object

In [16]:
dataset_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan"
test_dir = Path("/home/kartashov_ga/projects/tests/gsloc/26-04-22/MegaLoc/3rscan")
index_path = test_dir / "index"
query_cache_path = test_dir / "query_cache"
bench_report_dir = test_dir / "seq_benchmark_report"

In [17]:
from torchvision.transforms import functional as F

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True)
])

In [18]:
three_rscan_ds = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=index_path,
    rebuild_meta=False,  # meta.parquet already built
    # limit=20000,
    image_transform=image_transform_fn,
    save_meta=False,
    scene_filter_mode="listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
)
# You can create your own dataloader for index generation
# dataloader = DataLoader(
#     three_rscan_ds, batch_size=16, shuffle=False, num_workers=4, collate_fn=three_rscan_ds.collate_fn
# )

## Create model

In [19]:
model = MegaLoc()
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main


MegaLoc(
  (model): MegaLocModel(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, 

In [20]:

# # ``MultiModalVPRGraphEncoder`` layout matches ``best_model.pth``; MegaLoc weights are not in the ckpt.
# weights_path = Path("/home/kartashov_ga/projects/GSLoc/best_model.pth")
# ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)

# graph_enc = network.VPRGraphEncoder(
#     in_dim=4,
#     hidden_dim=256,
#     n_layers=1,
#     num_node_classes=528 + 1,
#     num_edge_classes=41,
#     node_emb_dim=64,
#     edge_emb_dim=64,
#     proj_dim=256,
# )
# model = network.MultiModalVPRGraphEncoder(
#     graph_encoder=graph_enc,
#     image_encoder=MegaLoc(),
#     image_out_dim=8448,
#     fusion_dim=8448,
#     mode="graph",
# )
# missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
# if unexpected:
#     raise RuntimeError(f"Unexpected checkpoint keys: {unexpected}")
# # ``missing`` lists ``image_encoder.*`` (hub MegaLoc), which the checkpoint does not store.

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()

## Create Index (files that are used to do retrievel based on database)

In [21]:
# generate function runs model for all dataset's elements and generates 3 files that are need for retrievel
index = FaissFlatIndex.generate(
    directory=index_path,
    dataset=three_rscan_ds,
    dataloader=None,
    model=model,
    rebuild_meta=True,
    rebuild_descriptors=False,
    batch_size = 24,
    num_workers = 6,
    shuffle = False,
    metric = "l2", # can be also "ip" - inner product
    version = 1)
print(f"Index created at {index_path}")
print(f"Index size: {index.size()}, dim: {index.dim()} metric: {index.metric()}")

2026-04-22 17:16:42.144 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 9,449 rows to /home/kartashov_ga/projects/tests/gsloc/26-04-22/MegaLoc/3rscan/index/meta.parquet
2026-04-22 17:16:42.145 | INFO     | mmpr.inference.index:generate:389 - meta.parquet file was saved in /home/kartashov_ga/projects/tests/gsloc/26-04-22/MegaLoc/3rscan/index
2026-04-22 17:16:42.145 | INFO     | mmpr.inference.index:generate:418 - Using existing descriptors.npy


2026-04-22 17:16:42.657 | INFO     | mmpr.inference.index:generate:436 - schema.json file was saved in /home/kartashov_ga/projects/tests/gsloc/26-04-22/MegaLoc/3rscan/index


Index created at /home/kartashov_ga/projects/tests/gsloc/26-04-22/MegaLoc/3rscan/index
Index size: 9449, dim: 8448 metric: l2


# Test PlaceRecognitionPipeline

In [22]:
pipeline = PlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
    # k=50,
)


seq_pr_pipeline = SequencePlaceRecognitionPipeline(
    index=index,
    model=model,
    device="cuda",
    max_window=25,
    per_frame_k=20,
    final_k=50,
    descriptor_agg="mean",
)

In [23]:
three_rscan_q = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=query_cache_path,
    # save_meta=True,
    rebuild_meta=False,
    # limit=10000,
    image_transform=image_transform_fn,
    scene_filter_mode="same_room_excluding_listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
)

In [24]:
inferencer = PRInferencer(
    pr_pipeline=pipeline,
    query_dataset=three_rscan_q,
    batch_size=16,
    num_workers=4,
    query_cache_dir=query_cache_path,
    k=25,
    device="cuda"
)

In [25]:
# frames = inferencer.run(rebuild_query_descriptors=True)
# inferencer.save(query_cache_path / "frames.npz", frames=frames)
frames = inferencer.load(query_cache_path / "frames.npz")

In [26]:
# inferencer.build_recall_benchmark_report(
#     database_dataset=three_rscan_ds,
#     ks=[1, 5, 10, 25],
#     similarity_kwargs={
#         "mode": "pose",
#         "trans_tol_m": 2,
#         "rot_tol_deg": 90
#         },
#     include_per_query=False
# )

In [27]:
# inferencer.build_recall_benchmark_report(
#     database_dataset=three_rscan_ds,
#     ks=[1, 5, 10, 25],
#     similarity_kwargs={
#         "mode": "room"
#     },
#     include_per_query=False
# )

In [ ]:
curr_report_dir = bench_report_dir / "room_k10"
result_df = inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    per_frame_k_used=10,
    save_dir=curr_report_dir,
    std_mode="scene",
    scene_df_field="scene",
)

  0%|          | 0/11 [00:00<?, ?it/s]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


  9%|▉         | 1/11 [00:52<08:40, 52.02s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 18%|█▊        | 2/11 [01:59<09:07, 60.83s/it]

...

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 73%|███████▎  | 8/11 [11:29<04:48, 96.15s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started


 82%|████████▏ | 9/11 [13:10<03:15, 97.79s/it]

fused rankings preparation started
rankings creation started
ranking iteration started


#Graph random report

In [15]:
def plot_metrics_vs_window_with_stats(
    summary_df,
    summary_all,
    metrics = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Plot per-map metrics vs w and overlay cross-map mean and weighted mean.

    Args:
        summary_df: DataFrame for a single map (has columns 'w' and metrics).
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    df = summary_df.sort_values("w").reset_index(drop=True)
    map_name = "all"
    # precompute simple mean once
    # group = summary_all.groupby("w", as_index=False)
    # mean_by_w = group[[m for m in metrics if m in summary_all.columns]].mean()

    for m in metrics:
        if m not in df.columns:
            continue
        fig = px.line(df, x="w", y=m, title=f"{map_name}: {m} vs sequence length (w)", markers=True)
        fig.update_layout(xaxis_title="sequence length (max_window)", yaxis_title=m)

        # Highlight maximum point on per-map line
        try:
            idx_max = df[m].astype(float).idxmax()
            w_star = int(df.loc[idx_max, "w"])  # sequence length at max
            y_star = float(df.loc[idx_max, m])
            fig.add_trace(
                go.Scatter(x=[w_star], y=[y_star], mode="markers", marker=dict(color="red", size=10), name="max", showlegend=False)
            )
            try:
                fig.add_vline(x=w_star, line_dash="dash", line_color="red")
            except Exception:
                fig.add_shape(type="line", x0=w_star, x1=w_star, y0=min(df[m].astype(float)), y1=max(df[m].astype(float)), line=dict(color="red", dash="dash"))
            fig.add_annotation(x=w_star, y=y_star, text=f"w={w_star}, {m}={y_star:.4f}", showarrow=True, arrowhead=2, ax=40, ay=-40)
        except Exception:
            pass

        # Overlay simple mean across maps
        # if m in mean_by_w.columns:
        #     fig.add_trace(
        #         go.Scatter(
        #             x=mean_by_w["w"],
        #             y=mean_by_w[m].astype(float),
        #             mode="lines",
        #             name="mean",
        #             line=dict(color="green", dash="dash"),
        #             showlegend=True,
        #         )
        #     )

        # Overlay weighted mean across maps (weights = num_valid per map)
        try:
            wmean_series = (
                summary_all
                .groupby("w")
                .apply(lambda g: float(np.average(g[m].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                .reset_index(name=m)
            )
            fig.add_trace(
                go.Scatter(
                    x=wmean_series["w"],
                    y=wmean_series[m].astype(float),
                    mode="lines",
                    name="weighted mean",
                    line=dict(color="purple", dash="dot"),
                    showlegend=True,
                )
            )
        except Exception:
            pass

        figs[m] = fig
        fig.show()
    return figs


### Base MegaLoc Results

In [16]:
plot_metrics_vs_window_with_stats(result_df, result_df)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQMFBwkLDQ8RExUX', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('Xm2HB39n7j+FXLrAWALuP1In84QFEO' ... 'dNX2iY7j/omfCqu5/uPytsdnr5pe4/'),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [23],
               'y': [0.9577605621192381]},
              {'line': {'color': 'purple', 'dash': 'dot'},
           

In [17]:
result_df_global_std = inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=range(1, 25, 2),
    per_frame_k_used=10,
    save_dir=query_cache_path / "seq_pr_benchmark",
    std_mode="global",
    scene_df_field="scene",
)

  0%|          | 0/12 [00:00<?, ?it/s]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


  8%|▊         | 1/12 [00:37<06:54, 37.66s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 17%|█▋        | 2/12 [01:43<09:00, 54.01s/it]

...

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 92%|█████████▏| 11/12 [14:45<01:27, 87.95s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started


100%|██████████| 12/12 [16:14<00:00, 81.22s/it]

fused rankings preparation started


In [18]:
plot_metrics_vs_window_with_stats(result_df_global_std, result_df_global_std)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQMFBwkLDQ8RExUX', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('Xm2HB39n7j+FXLrAWALuP1In84QFEO' ... 'dNX2iY7j/omfCqu5/uPytsdnr5pe4/'),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [23],
               'y': [0.9577605621192381]},
              {'line': {'color': 'purple', 'dash': 'dot'},
           

In [17]:
seq_inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
        },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:10<00:00, 1955.43it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,5938,0.282587,28.258697
1,5,21013,10549,0.502023,50.202256
2,10,21013,12317,0.586161,58.616095
3,25,21013,16555,0.787846,78.784562


In [ ]:
out = pipeline.infer(three_rscan_q[9000])

In [ ]:
out

PlaceRecognitionResult(descriptor=array([ 0.00143673,  0.00783881,  0.02143787, ...,  0.01978837,
       -0.00375643, -0.0045516 ], shape=(8448,), dtype=float32), indices=array([9000, 8787, 8786, 9001, 9005]), distances=array([8.8449399e-09, 5.0083816e-01, 7.0701253e-01, 7.5864244e-01,
       8.1422371e-01], dtype=float32), db_idx=array([9000, 8787, 8786, 9001, 9005]), db_pose=array([[ 0.798643  ,  0.983432  , -0.132033  ,  0.75250036, -0.5780175 ,
        -0.25381622, -0.18766014],
       [ 0.356377  ,  1.48875   , -0.131014  ,  0.8410546 , -0.427336  ,
        -0.3099954 , -0.11795727],
       [ 0.382755  ,  1.40634   , -0.0807452 ,  0.8403163 , -0.41481936,
        -0.32412416, -0.12937145],
       [ 0.841705  ,  0.995071  , -0.140846  ,  0.7504945 , -0.58146584,
        -0.24821363, -0.19247201],
       [ 0.860654  ,  0.989988  , -0.125784  ,  0.7427527 , -0.56332934,
        -0.28782725, -0.21939473]], dtype=float32))